In [0]:
%run ./00_config_setup

/home/spark-e33baba1-62c6-40f2-9966-3b/.ipykernel/1899/command-4887688419577413-2476527397:112: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TS         = datetime.utcnow()


In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {PLATINUM_SCHEMA}")

DataFrame[]

In [0]:
def write_platinum(
    df: DataFrame,
    table_name: str,
    s3_path: str,
    partition_cols: list = None,
    zorder_cols: list = None,
    label: str = ""
) -> int:
    """
    Full overwrite → register as Delta table → OPTIMIZE + ZORDER.
    Returns final row count.
    """
    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("path", s3_path)
    )
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)

    writer.saveAsTable(table_name)

    if zorder_cols:
        zorder_str = ", ".join(zorder_cols)
        spark.sql(f"OPTIMIZE {table_name} ZORDER BY ({zorder_str})")

    cnt = spark.table(table_name).count()
    tag = f"[{label}] " if label else ""
    print(f"  ✅ {tag}{table_name} → {cnt:,} rows written")
    return cnt

print("Loading Gold tables...")
df_fact        = spark.table(TBL_FACT_SALES)
df_dim_date    = spark.table(TBL_DIM_DATE)
df_dim_product = spark.table(TBL_DIM_PRODUCT)
df_dim_geo     = spark.table(TBL_DIM_GEOGRAPHY)
df_dim_store   = spark.table(TBL_DIM_STORE).filter(col("is_current") == True)
df_dim_customer= spark.table(TBL_DIM_CUSTOMER).filter(col("is_current") == True)
df_dim_domain  = spark.table(TBL_DIM_DOMAIN)


Loading Gold tables...


---
## AGG 1 — Executive Summary
**Dashboard tab**: Executive Summary  
**Grain**: domain × year × quarter  
**Metrics**: total_revenue, total_quantity, total_orders, avg_order_value, avg_review_score, unique_products, unique_customers

#### AGG 1 — Executive Summary

In [0]:
# Join fact → date dim to get year/quarter
df_exec = (
    df_fact
    .join(F.broadcast(df_dim_date.select("date_key", "year", "quarter", "quarter_name")),
          on="date_key", how="left")
    .join(F.broadcast(df_dim_domain.select(
              col("domain_key").alias("_dk"), col("domain_name").alias("domain_label"))),
          col("domain_key") == col("_dk"), "left")
    .groupBy("domain", "domain_label", "year", "quarter", "quarter_name")
    .agg(
        F.count("sales_key")                               .alias("total_orders"),
        F.sum("line_revenue")                              .alias("total_revenue"),
        F.sum("quantity")                                  .alias("total_quantity"),
        F.round(F.avg("line_revenue"), 2)                  .alias("avg_order_value"),
        F.round(F.avg(
            when(col("review_score").isNotNull(), col("review_score"))
        ), 4)                                              .alias("avg_review_score"),
        F.countDistinct("product_key")                     .alias("unique_products"),
        F.countDistinct(
            when(col("customer_key").isNotNull(), col("customer_key"))
        )                                                  .alias("unique_customers"),
        F.sum("helpful_votes")                             .alias("total_helpful_votes"),
        F.sum("total_votes")                               .alias("total_votes")
    )
    .withColumn("total_revenue",  F.round(col("total_revenue"), 2))
    .withColumn("platinum_ts",    current_timestamp())
)

write_platinum(
    df             = df_exec,
    table_name     = TBL_PLT_EXECUTIVE_SUMMARY,
    s3_path        = S3_PLT_EXECUTIVE_SUMMARY,
    partition_cols = ["domain"],
    zorder_cols    = ["year", "quarter"],
    label          = "Executive Summary"
)

print("\nPreview:")
display(
    spark.table(TBL_PLT_EXECUTIVE_SUMMARY)
    .orderBy("domain", "year", "quarter")
    .limit(10)
)

  ✅ [Executive Summary] retailflow360.platinum.agg_executive_summary → 0 rows written

Preview:


domain,domain_label,year,quarter,quarter_name,total_orders,total_revenue,total_quantity,avg_order_value,avg_review_score,unique_products,unique_customers,total_helpful_votes,total_votes,platinum_ts


---
## AGG 2 — Time Series
**Dashboard tab**: Time-Based Analysis  
**Grain**: domain × year × month  
**Metrics**: revenue, quantity, orders, MoM revenue delta, running totals  
**Note**: Window functions computed here so dashboard never re-calculates them.

#### AGG 2 — Time Series

In [0]:
df_time_base = (
    df_fact
    .join(F.broadcast(df_dim_date.select(
              "date_key", "year", "month", "month_name", "quarter", "quarter_name")),
          on="date_key", how="left")
    .groupBy("domain", "year", "month", "month_name", "quarter", "quarter_name")
    .agg(
        F.count("sales_key")                   .alias("total_orders"),
        F.round(F.sum("line_revenue"), 2)       .alias("total_revenue"),
        F.sum("quantity")                       .alias("total_quantity"),
        F.round(F.avg("line_revenue"), 2)       .alias("avg_order_value"),
        F.countDistinct("product_key")          .alias("unique_products"),
        F.round(F.avg(
            when(col("review_score").isNotNull(), col("review_score"))
        ), 4)                                   .alias("avg_review_score")
    )
)

# Window: month-over-month revenue change and running total — per domain
w_ts = (
    Window
    .partitionBy("domain")
    .orderBy("year", "month")
)

w_ts_running = (
    Window
    .partitionBy("domain")
    .orderBy("year", "month")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_time = (
    df_time_base
    .withColumn("prev_month_revenue",
        F.lag("total_revenue", 1).over(w_ts))
    .withColumn("mom_revenue_delta",
        F.round(
            when(col("prev_month_revenue").isNotNull() & (col("prev_month_revenue") > 0),
                (col("total_revenue") - col("prev_month_revenue")) /
                col("prev_month_revenue") * 100
            ).otherwise(lit(None).cast(DoubleType())),
        2))
    .withColumn("running_total_revenue",
        F.round(F.sum("total_revenue").over(w_ts_running), 2))
    .withColumn("running_total_orders",
        F.sum("total_orders").over(w_ts_running))
    .drop("prev_month_revenue")
    .withColumn("platinum_ts", current_timestamp())
)

write_platinum(
    df             = df_time,
    table_name     = TBL_PLT_TIME_SERIES,
    s3_path        = S3_PLT_TIME_SERIES,
    partition_cols = ["domain", "year"],
    zorder_cols    = ["month"],
    label          = "Time Series"
)

print("\nPreview:")
display(
    spark.table(TBL_PLT_TIME_SERIES)
    .orderBy("domain", "year", "month")
    .limit(12)
)

  ✅ [Time Series] retailflow360.platinum.agg_time_series → 0 rows written

Preview:


domain,year,month,month_name,quarter,quarter_name,total_orders,total_revenue,total_quantity,avg_order_value,unique_products,avg_review_score,mom_revenue_delta,running_total_revenue,running_total_orders,platinum_ts


---
## AGG 3 — Product Insights
**Dashboard tab**: Product Insights  
**Grain**: product_key × domain  
**Metrics**: revenue, quantity, orders, avg price, avg review score, rank by revenue within domain

#### AGG 3 — Product Insights

In [0]:
df_prod_agg = (
    df_fact
    .join(F.broadcast(
              df_dim_product.select("product_key", "product_name", "category", "domain")
              .withColumnRenamed("domain", "product_domain")),
          on="product_key", how="left")
    .groupBy("product_key", "product_name", "category", "domain", "product_domain")
    .agg(
        F.count("sales_key")                   .alias("total_orders"),
        F.round(F.sum("line_revenue"), 2)       .alias("total_revenue"),
        F.sum("quantity")                       .alias("total_quantity"),
        F.round(F.avg("unit_price"), 2)         .alias("avg_unit_price"),
        F.round(F.avg(
            when(col("review_score").isNotNull(), col("review_score"))
        ), 4)                                   .alias("avg_review_score"),
        F.count(
            when(col("review_score").isNotNull(), 1)
        )                                       .alias("review_count"),
        F.sum("helpful_votes")                  .alias("total_helpful_votes"),
        F.sum("total_votes")                    .alias("total_votes")
    )
)

w_prod_rank = Window.partitionBy("domain").orderBy(col("total_revenue").desc())

df_prod_insights = (
    df_prod_agg
    .withColumn("revenue_rank_in_domain",
        F.dense_rank().over(w_prod_rank))
    .withColumn("is_top_10",
        (col("revenue_rank_in_domain") <= 10).cast(BooleanType()))
    .withColumn("platinum_ts", current_timestamp())
)

write_platinum(
    df             = df_prod_insights,
    table_name     = TBL_PLT_PRODUCT_INSIGHTS,
    s3_path        = S3_PLT_PRODUCT_INSIGHTS,
    partition_cols = ["domain"],
    zorder_cols    = ["product_key", "category"],
    label          = "Product Insights"
)

print("\nTop 10 by revenue per domain:")
display(
    spark.table(TBL_PLT_PRODUCT_INSIGHTS)
    .filter(col("is_top_10") == True)
    .orderBy("domain", "revenue_rank_in_domain")
    .select("domain", "revenue_rank_in_domain", "product_name",
            "category", "total_revenue", "total_orders", "avg_review_score")
)

  ✅ [Product Insights] retailflow360.platinum.agg_product_insights → 0 rows written

Top 10 by revenue per domain:


domain,revenue_rank_in_domain,product_name,category,total_revenue,total_orders,avg_review_score


---
## AGG 4 — Geography & Stores
**Dashboard tab**: Geography & Stores  
**Grain**: geo_key × domain (× store_key for Liquor)  
**Metrics**: revenue, quantity, orders, lat/lon for map pins, store-level breakdown for Liquor

#### AGG 4 — Geography & Stores

In [0]:
df_geo_agg = (
    df_fact
    .filter(col("geo_key").isNotNull())
    .join(F.broadcast(
              df_dim_geo.select(
                  "geo_key", "city", "state", "zip_code",
                  "county", "lat", "lon", "domain"
              ).withColumnRenamed("domain", "geo_domain")),
          on="geo_key", how="left")
    .groupBy("geo_key", "city", "state", "zip_code",
             "county", "lat", "lon", "domain", "geo_domain")
    .agg(
        F.count("sales_key")               .alias("total_orders"),
        F.round(F.sum("line_revenue"), 2)   .alias("total_revenue"),
        F.sum("quantity")                   .alias("total_quantity"),
        F.round(F.avg("unit_price"), 2)     .alias("avg_unit_price"),
        F.countDistinct("product_key")      .alias("unique_products")
    )
    .withColumn("platinum_ts", current_timestamp())
)

write_platinum(
    df             = df_geo_agg,
    table_name     = TBL_PLT_GEOGRAPHY,
    s3_path        = S3_PLT_GEOGRAPHY,
    partition_cols = ["domain"],
    zorder_cols    = ["state", "city"],
    label          = "Geography"
)

print("\nTop cities by revenue:")
display(
    spark.table(TBL_PLT_GEOGRAPHY)
    .orderBy(col("total_revenue").desc())
    .select("domain", "city", "state", "county",
            "total_revenue", "total_orders", "lat", "lon")
    .limit(10)
)

  ✅ [Geography] retailflow360.platinum.agg_geography → 0 rows written

Top cities by revenue:


domain,city,state,county,total_revenue,total_orders,lat,lon


---
## AGG 5 — Customer Insights
**Dashboard tab**: Customer Insights  
**Grain**: customer_key (Books domain only — Electronics/Liquor have no customer)  
**Metrics**: total spend, review count, avg score, helpfulness ratio, activity span

#### AGG 5 — Customer Insights

In [0]:
df_cust_agg = (
    df_fact
    .filter(
        col("customer_key").isNotNull() &
        (col("domain") == "Books")
    )
    .join(F.broadcast(
              df_dim_customer.select(
                  "customer_key", "user_id", "profile_name", "is_anonymous")
              ),
          on="customer_key", how="left")
    .join(F.broadcast(
              df_dim_date.select("date_key", "year", "month")),
          on="date_key", how="left")
    .groupBy("customer_key", "user_id", "profile_name", "is_anonymous", "domain")
    .agg(
        F.count("sales_key")                           .alias("total_reviews"),
        F.round(F.sum("line_revenue"), 2)               .alias("total_spend"),
        F.round(F.avg(
            when(col("review_score").isNotNull(), col("review_score"))
        ), 4)                                           .alias("avg_review_score"),
        F.countDistinct("product_key")                  .alias("unique_titles_reviewed"),
        F.sum("helpful_votes")                          .alias("total_helpful_votes"),
        F.sum("total_votes")                            .alias("total_votes"),
        F.min(col("year") * 100 + col("month"))        .alias("first_activity_ym"),
        F.max(col("year") * 100 + col("month"))        .alias("last_activity_ym"),
        F.countDistinct(col("year") * 100 + col("month")).alias("active_months")
    )
)

# Helpfulness ratio — helpful / total
df_cust_insights = (
    df_cust_agg
    .withColumn(
        "helpfulness_ratio",
        when(
            col("total_votes").isNotNull() & (col("total_votes") > 0),
            F.round(
                col("total_helpful_votes").cast(DoubleType()) /
                col("total_votes").cast(DoubleType()), 4
            )
        ).otherwise(lit(None).cast(DoubleType()))
    )
    # Segment: casual (<5 reviews), regular (5-20), power (>20)
    .withColumn(
        "reviewer_segment",
        when(col("total_reviews") >= 20, lit("Power Reviewer"))
        .when(col("total_reviews") >= 5,  lit("Regular Reviewer"))
        .otherwise(lit("Casual Reviewer"))
    )
    .withColumn("platinum_ts", current_timestamp())
)

write_platinum(
    df             = df_cust_insights,
    table_name     = TBL_PLT_CUSTOMER_INSIGHTS,
    s3_path        = S3_PLT_CUSTOMER_INSIGHTS,
    partition_cols = ["domain"],
    zorder_cols    = ["customer_key", "reviewer_segment"],
    label          = "Customer Insights"
)

print("\nSegment breakdown:")
spark.table(TBL_PLT_CUSTOMER_INSIGHTS) \
    .groupBy("reviewer_segment") \
    .agg(
        F.count("customer_key").alias("customers"),
        F.round(F.avg("total_reviews"), 1).alias("avg_reviews"),
        F.round(F.avg("avg_review_score"), 2).alias("avg_score")
    ) \
    .orderBy(col("avg_reviews").desc()) \
    .show()

print("\nPreview:")
display(
    spark.table(TBL_PLT_CUSTOMER_INSIGHTS)
    .filter(col("is_anonymous") == False)
    .orderBy(col("total_reviews").desc())
    .limit(10)
)

  ✅ [Customer Insights] retailflow360.platinum.agg_customer_insights → 0 rows written

Segment breakdown:
+----------------+---------+-----------+---------+
|reviewer_segment|customers|avg_reviews|avg_score|
+----------------+---------+-----------+---------+
+----------------+---------+-----------+---------+


Preview:


customer_key,user_id,profile_name,is_anonymous,domain,total_reviews,total_spend,avg_review_score,unique_titles_reviewed,total_helpful_votes,total_votes,first_activity_ym,last_activity_ym,active_months,helpfulness_ratio,reviewer_segment,platinum_ts
